# The terminal interface

Every earlier notebook calls PyPFT from code. PyPFT also has a terminal interface that does the most common job without code: load a polar array from a file, run the PFT or the IPFT on it, and view or save the result's figures. It is a thin front end over the functions the earlier notebooks use and computes nothing of its own, so everything it does can also be done in a script. It runs in any terminal and needs no graphical toolkit.

This notebook describes how to launch it, its screens, the files it reads, and how it names the figures it saves. It ends by driving the interface from code, on the maze of [08_visualization](08_visualization.ipynb).

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# Paths and temporary directories, and matplotlib for figures
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt

# Textual draws the interface; the code below types into its fields and presses
# its buttons
from textual.widgets import Button, Input

# PyPFT's domains, and its terminal interface
import pypft
from pypft.cli import FigurePart, PyPFTApp, output_filename

## Launching it

From a terminal, in an environment where PyPFT is installed, either command opens the interface:

```bash
python -m pypft
pypft
```

It takes over the whole terminal until **Quit** is pressed. The mouse and the keyboard both work: `Tab` moves between fields and buttons, and `Enter` presses a button.

## The screens

The interface walks through one screen per step. Every screen has a **Quit** button and a status line, which shows the last result or error; every screen but the first has a **Back** button. An error, such as a file of the wrong shape, is shown on the status line, and the interface keeps running.

1. **Start**: transform a file, or open the tutorials. The tutorials open online in a browser; from a clone of the repository with the development tools installed, the notebooks can also be opened in JupyterLab.
2. **Layout**: whether the file holds one polar array or a batch of them ([06_batches](06_batches.ipynb)), or holds radial k-space lines in one of the layouts `RadialLineLayout` names ([09_experimental_data](09_experimental_data.ipynb)).
3. **File**: the file's path, typed or picked in a file dialog that lists only the files the layout of step 2 is read from. A path pasted into the dialog's own field opens a directory there, or picks a file of those types. The file is read and checked against the choice of step 2. For radial lines in a `.npz` or `.mat` file, which hold named arrays, an **Array** screen comes next to choose the array holding the lines; a `.npy` file's single array is read as it is. Then an **Acquisition** screen comes, with one field per axis the lines are recorded over (slices, phases, coils, ...), each taking an index or `all`, as `select_radial_kspace` takes them. Every field starts at index 0; one index of every axis reads a single sample, and `all` keeps an axis whole in a batch.
4. **Grid**: where the samples sit (see below), the grid's space limit $R$, whether $R$ is a space limit or a band limit, and optionally a band limit to check the Nyquist condition against. The adequacy checks of [02_sampling_grids](02_sampling_grids.ipynb) run here, and the next screen shows their warnings. For k-space, the screen fills in the radius and proposes a grid: $R$ is the field of view the samples resolve, the grid has a quarter of the data's spokes, and as many radial samples as keep its frequencies inside the data.
5. **Transform**: the PFT (the file holds spatial samples) or the IPFT (the file holds frequency samples), the path, and whether to trace every domain. The **exact** path is the default: `forward_pft`/`inverse_pft`, exactly invertible. The **accurate** path is the ring-consistent route of [05_pft_and_ipft](05_pft_and_ipft.ipynb), whose inverse is an approximation. Tracing, on the exact path only, keeps all four domains of [04_domains](04_domains.ipynb) instead of the result alone.
6. **Result**: **View** draws one sample's magnitude and phase with `plot_signal` ([08_visualization](08_visualization.ipynb)), and **Save** writes every kept domain's figures to a directory. For a batch, a batch-element field picks the sample to view, and every element is drawn into the same figure, replacing the previous one. The figure opens in a matplotlib window when matplotlib's backend can open one; otherwise it is saved to a temporary file whose path the status line shows.

## Input files

The interface reads polar arrays in PyPFT's layout, `(radial, angular)` for a single sample and `(radial, angular, batch)` for a batch, with a centered angular axis ([01_polar_images_and_grids](01_polar_images_and_grids.ipynb)). A file does not record which grid its samples sit on, so the grid screen asks for it.

- **`.npy`**: any array saved with `numpy.save`, real or complex, a single sample or a batch. This is the format for frequency-domain input to the IPFT, and for batches.
- **`.npz`**: uniform polar k-space saved by `RadialKSpace.save` ([09_experimental_data](09_experimental_data.ipynb)). It carries the frequency radius its samples cover, which the grid screen fills in, and it is transformed with the IPFT.
- **Radial k-space lines**, read when their layout is chosen on the layout screen: a layout describes the array only, so the lines may be a `.npy` file's array or one named array of a `.npz` or `.mat` file, such as the committed excerpt `tests/samples/P01_slc1_phs1_lins196_cols416_cha4.mat`. The selection chosen on the acquisition screen is converted into uniform polar k-space, which is then treated like a `.npz` file of `RadialKSpace.save`.
- **`.tif`, `.tiff`, `.png`**: a grayscale image holding one real spatial sample, such as the committed maze `tests/samples/maze_polar.tif`. An image cannot hold complex values, so it is input to the PFT only.

## On the transform's grid, or uniform polar

[02_sampling_grids](02_sampling_grids.ipynb) shows that the transform's own grid, `PolarGrid`, is not a uniform polar grid: each spoke has its own radii. The grid screen therefore asks where the file's samples sit.

- **On the transform's grid**: the samples are at `grid.r` (or `grid.rho` for frequency samples), like the maze. The array's shape fixes the grid's size, so only $R$ is asked for, and the array is transformed as it is.
- **Uniform polar**: the same equally spaced radii on every spoke, like the output of `cartesian_to_polar`. The grid's size and the radius the samples cover are asked for as well. On the exact path, the array is first interpolated along each spoke onto the grid with `resample_uniform_polar` (or `resample_uniform_polar_frequency` for frequency samples), and the result screen says so, since the result then carries that interpolation's error. The accurate path reads uniform polar samples directly.

## Saved figures and their names

**Save** writes one `.png` file per kept domain, per image of `plot_signal`'s magnitude and phase pair, and per batch element. Each file is named `<stem>_<domain>_<magnitude|phase>[_b<NNN>].png`: `<stem>` is the input file's name without its suffix, `<domain>` names the domain with its coordinate system (`polar_frequency_angular`, ...), and for a batch `_b<NNN>` is the batch element, padded with zeros to three digits, or to as many as the largest element needs, so the files sort in order. The interface never replaces an existing file without asking first. `output_filename` builds these names:

In [ ]:
# The two files of a single sample's frequency-domain result
for part in FigurePart:
    print(
        output_filename(
            stem="maze_polar", domain=pypft.PolarDomain.FREQUENCY_ANGULAR, part=part
        )
    )

# A batch element's file: three digits, or more for a batch of more than 1000
for batch_size in (12, 2500):
    print(
        output_filename(
            stem="scans",
            domain=pypft.PolarDomain.SPATIAL_ANGULAR,
            part=FigurePart.MAGNITUDE,
            batch_index=7,
            batch_size=batch_size,
        )
    )

## Driving the interface from code

Textual, which draws the interface, can also run it without a terminal and press its buttons from code. The cell below uses that to walk through every screen: it loads the maze, which sits on its own grid with $R = 1$, runs the PFT on the exact path, and saves the result's figures. The directory is temporary; change `output_directory` to the directory you want the figures saved in.

The maze's grid has few radial samples for its 48 spokes, so the grid's adequacy check warns, and the transform screen shows that warning. The interface draws its screens instead of printing, so the cell keeps what the screens say and prints it after the app has quit.

In [ ]:
# Change this to the directory you want the figures saved in
output_directory = Path(tempfile.mkdtemp(prefix="pypft_interface_"))
maze_path = Path("../tests/samples/maze_polar.tif")


async def press(pilot, button_id):
    # Press one of the current screen's buttons, then wait until the app reacts:
    # it opens another screen or shows something new on its status line, other
    # than a message ending in "..." that a long step shows while it runs
    app = pilot.app
    screen, status = app.screen, app.status
    app.screen.query_one(selector=f"#{button_id}", expect_type=Button).press()
    while (app.screen is screen and app.status == status) or app.status.endswith(
        "..."
    ):
        await pilot.pause(delay=0.05)


def type_into(pilot, field_id, text):
    # Type text into one of the current screen's fields
    pilot.app.screen.query_one(selector=f"#{field_id}", expect_type=Input).value = text


app = PyPFTApp()
async with app.run_test(size=(110, 70)) as pilot:
    # Start: transform a file; a single sample is the default choice
    await press(pilot=pilot, button_id="transform")
    await press(pilot=pilot, button_id="next")

    # File: read the maze
    type_into(pilot=pilot, field_id="path", text=str(maze_path))
    await press(pilot=pilot, button_id="load")

    # Grid: the maze sits on the transform's grid, the default choice, with R = 1
    type_into(pilot=pilot, field_id="R", text="1.0")
    await press(pilot=pilot, button_id="next")
    grid_notices = app.screen.notices

    # Transform: the PFT on the exact path, the default choices
    await press(pilot=pilot, button_id="run")

    # Result: save every figure
    type_into(pilot=pilot, field_id="directory", text=str(output_directory))
    await press(pilot=pilot, button_id="save")
    save_status = app.status

    # Quit
    app.screen.query_one(selector="#quit", expect_type=Button).press()

# What the transform screen said about the grid, and the result screen about saving;
# printed here, since the app captures printing while it runs
print(grid_notices)
print(save_status)

The directory now holds the two figures of the single result, named after the input file and the frequency domain:

In [ ]:
# List the saved files
saved = sorted(output_directory.iterdir())
for path in saved:
    print(path.name)

# Show the saved magnitude figure as an image
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(X=plt.imread(fname=saved[0]))
ax.set_title(label=saved[0].name)
ax.set_axis_off()
plt.show()